In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.embedding


>DOM embedding
output-file: core.dom.embedding
title: core.dom.embedding

This notebook provides utility functions for DOM embedding
---

In [ ]:
# | default_exp core.dom.embedding

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
import os
import re
from collections.abc import Callable
from pathlib import Path
from typing import Optional, Any
import hashlib

import numpy as np
from ollama import AsyncClient

from chromadb.api import ClientAPI
from chromadb.api.models.Collection import Collection

In [ ]:
# | hide
import asyncio
import threading

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide

def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, object] = {}
    error: dict[str, BaseException] = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')

class FakeEmbedResponse:
    def __init__(self, embeddings=None, model="fake-model"):
        self.embeddings = embeddings
        self.model = model


class FakeAsyncClient:
    def __init__(self, responses):
        self.responses = list(responses)
        self.calls = []

    async def embed(self, model: str, input: str):
        self.calls.append({"model": model, "input": input})
        if not self.responses:
            raise AssertionError("No fake responses remaining")
        response = self.responses.pop(0)
        return response() if callable(response) else response


class FakeCollection:
    def __init__(self):
        self.records = []

    def add(self, ids, metadatas, embeddings, documents):
        self.records.append(
            {
                "ids": ids,
                "metadatas": metadatas,
                "embeddings": embeddings,
                "documents": documents,
            }
        )

In [ ]:
# | export
def normalize_embeddings(raw_embeddings: list[list[float]] | list[float] | None) -> list[list[float]] | None:
    """Normalize Ollama embedding payloads to Chroma-compatible list[list[float]]."""
    if raw_embeddings is None:
        return None
    if isinstance(raw_embeddings, dict):
        raw_embeddings = raw_embeddings.get('embeddings') or raw_embeddings.get('embedding')
    # if hasattr(raw_embeddings, 'tolist'):
    #     raw_embeddings = raw_embeddings.tolist()
    if isinstance(raw_embeddings, list):
        if not raw_embeddings:
            return None
        if isinstance(raw_embeddings[0], (int, float)):
            return [[float(x) for x in raw_embeddings]]
        if isinstance(raw_embeddings[0], list):
            normalized = []
            for row in raw_embeddings:
                if isinstance(row, list) and row:
                    normalized.append([float(x) for x in row])
            return normalized or None
    return None


In [ ]:
# | hide
def test_normalize_embeddings_handles_supported_shapes():
    test_eq(normalize_embeddings(None), None)
    test_eq(normalize_embeddings([]), None)
    test_eq(normalize_embeddings([1, 2.5, 3]), [[1.0, 2.5, 3.0]])
    test_eq(normalize_embeddings([[1, 2], [3.5, 4]]), [[1.0, 2.0], [3.5, 4.0]])
    test_eq(normalize_embeddings({"embedding": [1, 2]}), [[1.0, 2.0]])
    test_eq(normalize_embeddings({"embeddings": [[1, 2], [], [3, 4]]}), [[1.0, 2.0], [3.0, 4.0]])
    test_eq(normalize_embeddings("invalid"), None)


test_normalize_embeddings_handles_supported_shapes()

In [ ]:
# | export
async def get_embeddings_with_retry(text: str, item_label: str, client:AsyncClient, model:str="bge-m3") -> tuple[list[float] | None, Any]:
    """Fetch embeddings and retry once if payload is empty/invalid.

    Args:
        text: The input text to embed.
        item_label: A label for the item being embedded, used for logging.
        client: The AsyncClient instance to use for embedding.
        model: The embedding model to use.

    Returns:
        A tuple of (embeddings, response). If embeddings are empty, returns (None, response).
    """
    response = await client.embed(model=model, input=text)
    raw_embeddings: list[list[float]] | list[float] | None = None

    if isinstance(response, dict):
        candidate = response.get("embeddings") or response.get("embedding")
        if isinstance(candidate, list):
            raw_embeddings = candidate
    elif isinstance(response, list):
        raw_embeddings = response
    else:
        candidate = getattr(response, "embeddings", None)
        if isinstance(candidate, list):
            raw_embeddings = candidate

    embeddings = normalize_embeddings(raw_embeddings)
    if embeddings:
        return embeddings[0], response

    # Small retry for transient empty payloads
    response_retry = await client.embed(model=model, input=text)
    raw_embeddings_retry: list[list[float]] | list[float] | None = None

    if isinstance(response_retry, dict):
        candidate_retry = response_retry.get("embeddings") or response_retry.get("embedding")
        if isinstance(candidate_retry, list):
            raw_embeddings_retry = candidate_retry
    elif isinstance(response_retry, list):
        raw_embeddings_retry = response_retry
    else:
        candidate_retry = getattr(response_retry, "embeddings", None)
        if isinstance(candidate_retry, list):
            raw_embeddings_retry = candidate_retry

    normalized_retry = normalize_embeddings(raw_embeddings_retry)
    embeddings_retry = normalized_retry[0] if normalized_retry else None
    if embeddings_retry:
        print(f"Embedding retry succeeded for {item_label}")
        return embeddings_retry, response_retry

    print(f"Embedding retry failed for {item_label}; payload remains empty/invalid")
    return None, response_retry

In [ ]:
# | hide
def test_get_embeddings_with_retry_returns_first_valid_payload():
    client = FakeAsyncClient([
        {"embeddings": [[0.1, 0.2, 0.3]]},
    ])

    embeddings, response = run_async(
        get_embeddings_with_retry("hello", "item-1", client=client, model="m1")  # type: ignore
    )

    test_eq(embeddings, [0.1, 0.2, 0.3])
    test_eq(response["embeddings"], [[0.1, 0.2, 0.3]])
    test_eq(len(client.calls), 1)


def test_get_embeddings_with_retry_retries_once_then_succeeds():
    client = FakeAsyncClient([
        {"embedding": []},
        FakeEmbedResponse([[1, 2, 3]], model="retry-model"),
    ])

    embeddings, response = run_async(
        get_embeddings_with_retry("hello", "item-2", client=client, model="m2")  # type: ignore
    )

    test_eq(embeddings, [1.0, 2.0, 3.0])
    test_eq(response.model, "retry-model")
    test_eq(len(client.calls), 2)


def test_get_embeddings_with_retry_returns_none_after_failed_retry():
    client = FakeAsyncClient([
        {"embeddings": []},
        FakeEmbedResponse(None),
    ])

    embeddings, response = run_async(
        get_embeddings_with_retry("hello", "item-3", client=client, model="m3")  # type: ignore
    )

    test_eq(embeddings, None)
    test_eq(response.embeddings, None)
    test_eq(len(client.calls), 2)


test_get_embeddings_with_retry_returns_first_valid_payload()
test_get_embeddings_with_retry_retries_once_then_succeeds()
test_get_embeddings_with_retry_returns_none_after_failed_retry()

In [ ]:
# | export
async def embed(
    semantics_json: str,
    file_path: Path,
    embed_types: set[str],
    db_client: Optional[ClientAPI],
    collection: Collection,
    llm_client:AsyncClient,
    model:str="bge-m3",
    action: Optional[Callable] = None,
) -> str:
    """Walk through the semantics AST and embed supported nodes."""
    if action is None:
        action = lambda node: node  # type: ignore  # noqa: E731

    ast = json.loads(semantics_json)
    assert isinstance(ast, dict), f"AST should be a dictionary, got {type(ast)}"

    canonical_json = json.dumps(
        ast, sort_keys=True, ensure_ascii=False, separators=(",", ";")
    ).encode("utf-8").decode("utf-8")
    full_canonical_string = str(file_path) + canonical_json

    # Create a unique ID for the AST based on its content
    ast["i"] = hashlib.sha256(full_canonical_string.encode("utf-8")).hexdigest()
    cur_object_path = [ast["i"]]

    ast_metadata: dict[str, str | int | float | bool] = {
        "embed_model": model,
    }
    ast["m"] = ast_metadata
    ast_embeddings, ast_response = await get_embeddings_with_retry(
        text=ast.get("s", ""),
        item_label=f"AST {ast['i']}",
        client=llm_client,
        model=model,
    )

    if not ast_embeddings:
        print(
            "Skip AST embedding add due to empty/invalid embeddings: "
            f"{type(getattr(ast_response, 'embeddings', ast_response))}"
        )
    else:
        try:
            embeddings_array = np.asarray(ast_embeddings, dtype=np.float32)
            if embeddings_array.ndim == 1:
                embeddings_array = embeddings_array.reshape(1, -1)

            collection.add(
                ids=[ast["i"]],
                metadatas=[ast_metadata],
                embeddings=embeddings_array,
                documents=[ast.get("s", "")],
            )
        except Exception as e:
            print(f"Failed to add AST to collection: {e}")

    ast = await embed_walk_node(
        node=ast, 
        cur_object_path=cur_object_path,
        collection=collection,
        embed_types=embed_types,
        llm_client=llm_client,
        model=model,
        action=action
        )
    embed_json = json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")

    return embed_json

In [ ]:
# | hide
def test_embed_adds_ast_and_nested_nodes_then_returns_json():
    semantics = {
        "t": "Document",
        "s": "document summary",
        "children": [
            {"t": "Section", "s": "section summary", "children": []}
        ],
    }
    collection = FakeCollection()
    client = FakeAsyncClient([
        FakeEmbedResponse([[9, 9]], model="ast-model"),
        FakeEmbedResponse([[1, 1]], model="section-model"),
    ])

    result_json = run_async(
        embed(
            semantics_json=json.dumps(semantics),
            file_path=Path("sample.md"),
            embed_types={"Section"},
            db_client=None,
            collection=collection,
            llm_client=client,
            model="seed-model",
            action=lambda node: {**node, "seen": True} if isinstance(node, dict) else node,
        )
    )

    result = json.loads(result_json)
    test_eq(result["m"]["embed_model"], "seed-model")
    test_eq(result["m"]["seen"], True)
    test_eq(result["seen"], True)
    test_eq(result["children"][0]["seen"], True)
    test_eq(len(result["i"]), 64)
    test_eq(len(result["children"][0]["i"]), 64)
    test_eq(len(collection.records), 2)
    test_eq(collection.records[0]["documents"], ["document summary"])
    test_eq(collection.records[1]["documents"], ["section summary"])
    test_eq(client.calls[0]["input"], "document summary")
    test_eq(client.calls[1]["input"], "section summary")

In [ ]:
# | export
async def embed_node(node: dict, collection:Collection, cur_object_path:list[str],embed_types:set[str], llm_client:AsyncClient, model: str) -> dict:
    """embeds the node summary text"""
    # nonlocal collection, cur_object_path  # declare the collection variable in the closure
    
    assert node.get('t') in embed_types, f"Node type {node.get('t')} is not in embed_types: {embed_types}"
    # assert node.get('s'), f"Node summary text is empty for node: {node}"
    
    canonical_string = json.dumps(node, 
                                sort_keys=True, 
                                ensure_ascii=False,
                                separators=(',',';')
                            ).encode("utf-8").decode("utf-8")
    # Create a unique ID for the node based on its content
    full_canonical_string = str(cur_object_path) + canonical_string

    # Create the identifier for the node
    # The identifier is a SHA256 hash of the full canonical string
    # This ensures that the identifier is unique and consistent for the same content
    node['i'] = hashlib.sha256(full_canonical_string.encode("utf-8")).hexdigest()
    # Create the embedding for the node summary
    txt = node.get('s', '')
    if txt != '':
        node_embeddings, node_response = await get_embeddings_with_retry(txt, f"node {node.get('i', '')}", client=llm_client,model=model)
        model = node_response.model if isinstance(node_response.model, str) and node_response.model else model
        node_metadata: dict[str, str | int | float | bool] = {
            # 'obj_path': cur_object_path,  # type: ignore
            'embed_model': model,
        }
        node['m'] = node_metadata
    else:
        node_embeddings = None
        node_metadata = dict()

    if not node_embeddings:
        print(f"Skip node embedding add due to empty/invalid embeddings for node {node.get('i', '')}")
    else:
        try:
            assert node_embeddings, "node embedding should not be empty at this point"
            embeddings_array = np.asarray(node_embeddings, dtype=np.float32)
            if embeddings_array.ndim == 1:
                embeddings_array = embeddings_array.reshape(1, -1)

            collection.add(
                ids=[node["i"]],
                metadatas=[node_metadata],
                embeddings=embeddings_array,
                documents=[node.get("s", "")],
            )
        except Exception as e:
            print(f"Failed to add node to collection: {e}")

    return node

In [ ]:
# | hide
def test_embed_node_adds_collection_record_for_supported_node():
    node = {"t": "Section", "s": "section summary"}
    collection = FakeCollection()
    client = FakeAsyncClient([FakeEmbedResponse([[0.5, 1.5]], model="embed-model")])

    result = run_async(
        embed_node(
            node=node,
            collection=collection,
            cur_object_path=["root-id"],
            embed_types={"Section"},
            llm_client=client,
            model="seed-model",
        )
    )

    test_eq(result["s"], "section summary")
    test_eq(result["m"], {"embed_model": "embed-model"})
    test_eq(len(result["i"]), 64)
    test_eq(len(collection.records), 1)
    test_eq(collection.records[0]["documents"], ["section summary"])
    test_eq(collection.records[0]["metadatas"], [{"embed_model": "embed-model"}])
    test_eq(collection.records[0]["embeddings"].shape, (1, 2))


def test_embed_node_skips_empty_summary_and_rejects_wrong_type():
    collection = FakeCollection()
    client = FakeAsyncClient([])
    empty_node = {"t": "Section", "s": ""}

    result = run_async(
        embed_node(
            node=empty_node,
            collection=collection,
            cur_object_path=["root-id"],
            embed_types={"Section"},
            llm_client=client,
            model="seed-model",
        )
    )

    test_eq(result.get("m"), None)
    test_eq(len(collection.records), 0)
    test_eq(len(client.calls), 0)
    test_fail(
        lambda: run_async(
            embed_node(
                node={"t": "Para", "s": "nope"},
                collection=collection,
                cur_object_path=["root-id"],
                embed_types={"Section"},
                llm_client=client,
                model="seed-model",
            )
        ),
        contains="Node type Para is not in embed_types",
    )


test_embed_node_adds_collection_record_for_supported_node()
test_embed_node_skips_empty_summary_and_rejects_wrong_type()

In [ ]:
# | export
async def embed_walk_node(node: dict|list, cur_object_path: list[str],collection:Collection, embed_types:set[str], llm_client:AsyncClient, model:str="bge-m3", action:Optional[Callable]=None) -> dict|list:
    # nonlocal cur_object_path

    if action is not None:
        node = action(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    await embed_walk_node(child, cur_object_path, collection, embed_types, llm_client=llm_client, model=model, action=action)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                if value.get('t') in embed_types:
                    value = await embed_node(value, collection, cur_object_path, embed_types, llm_client=llm_client, model=model)  # Embed the node summary text
                node[key] = await embed_walk_node(value, cur_object_path, collection, embed_types, llm_client=llm_client, model=model, action=action)
        # embed the summary at last!
        if node.get('t') in embed_types:
            node = await embed_node(node,
                                    collection, cur_object_path, embed_types,llm_client=llm_client,model=model)  # Embed the node summary text
            cur_object_path.append(node.get('i', ''))  # Add the node ID to the current object path
    elif isinstance(node, list):
        node = [
            await embed_walk_node(child, cur_object_path, collection, embed_types, llm_client=llm_client, model=model, action=action) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node 


In [ ]:
# | hide
def test_embed_walk_node_recurses_lists_and_applies_action():
    semantics = {
        "t": "Section",
        "s": "root summary",
        "children": [
            {"t": "Table", "s": "table summary", "rows": []},
            {"plain": "value"},
        ],
    }
    collection = FakeCollection()
    client = FakeAsyncClient([
        FakeEmbedResponse([[1, 0]], model="walk-model"),
        FakeEmbedResponse([[0, 1]], model="walk-model"),
    ])

    result = run_async(
        embed_walk_node(
            node=semantics,
            cur_object_path=["ast-id"],
            collection=collection,
            embed_types={"Section", "Table"},
            llm_client=client,
            model="seed-model",
            action=lambda node: {**node, "visited": True} if isinstance(node, dict) else node,
        )
    )

    test_eq(result["visited"], True)
    test_eq(result["children"][0]["visited"], True)
    test_eq(result["children"][1]["visited"], True)
    test_eq(len(collection.records), 2)
    test_eq(collection.records[0]["documents"], ["table summary"])
    test_eq(collection.records[1]["documents"], ["root summary"])


test_embed_adds_ast_and_nested_nodes_then_returns_json()
test_embed_walk_node_recurses_lists_and_applies_action()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()